# 08 — Model Explainability (SHAP)
Global summary plot + local force plot for a specific employee.

In [1]:

import pandas as pd
import numpy as np
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("SHAP not installed — will create summary without SHAP plots")

PROC = r'../data/processed'
MODELS = r'../models'

df = pd.read_csv(f'{PROC}/feature_matrix.csv')
target = 'AttritionRisk_Label'
drop_cols = [c for c in [target, 'EmployeeID'] if c in df.columns]
X = df.drop(columns=drop_cols).astype(float)
y = df[target]

pipeline = joblib.load(f'{MODELS}/attrition_pipeline.joblib')
print(f"Loaded pipeline: {pipeline.named_steps}")


Loaded pipeline: {'sc': StandardScaler(), 'clf': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)}


In [2]:

if HAS_SHAP:
    # Get the classifier step from pipeline
    # Transform X through all steps except classifier
    clf_step = list(pipeline.named_steps.keys())[-1]
    pre_steps = list(pipeline.named_steps.keys())[:-1]
    
    X_transformed = X.copy()
    for step in pre_steps:
        X_transformed = pipeline.named_steps[step].transform(X_transformed)
    X_transformed = pd.DataFrame(X_transformed, columns=X.columns)
    
    clf = pipeline.named_steps[clf_step]
    
    # Use TreeExplainer for tree models, LinearExplainer for LR
    clf_name = type(clf).__name__
    print(f"Classifier type: {clf_name}")
    
    if 'Forest' in clf_name or 'XGB' in clf_name or 'Boost' in clf_name:
        explainer = shap.TreeExplainer(clf)
        shap_values = explainer.shap_values(X_transformed)
        if isinstance(shap_values, list):
            shap_vals = shap_values[1]  # class 1 (attrition)
        else:
            shap_vals = shap_values
    else:
        explainer = shap.LinearExplainer(clf, X_transformed)
        shap_values = explainer.shap_values(X_transformed)
        shap_vals = shap_values
    
    print(f"SHAP values shape: {shap_vals.shape}")
else:
    print("Skipping SHAP (not installed)")
    shap_vals = None
    X_transformed = X


Classifier type: XGBClassifier
SHAP values shape: (500, 43)


In [3]:

if HAS_SHAP and shap_vals is not None:
    # ── Global Summary Plot ──
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_vals, X_transformed, plot_type='bar', show=False,
                      max_display=15)
    plt.title('Global Feature Importance (SHAP)', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{MODELS}/shap_global_summary.png', dpi=100, bbox_inches='tight')
    plt.close()
    print("Saved: shap_global_summary.png")
    
    # ── Dot summary plot ──
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_vals, X_transformed, show=False, max_display=15)
    plt.title('SHAP Feature Impact Distribution', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{MODELS}/shap_dot_summary.png', dpi=100, bbox_inches='tight')
    plt.close()
    print("Saved: shap_dot_summary.png")
else:
    # Create a feature importance chart using model's built-in importances
    clf_step = list(pipeline.named_steps.keys())[-1]
    clf = pipeline.named_steps[clf_step]
    if hasattr(clf, 'feature_importances_'):
        importances = pd.Series(clf.feature_importances_, index=X.columns)
        top15 = importances.nlargest(15)
        fig, ax = plt.subplots(figsize=(10, 8))
        top15.sort_values().plot(kind='barh', ax=ax, color='steelblue')
        ax.set_title('Feature Importances (Model Built-in)')
        plt.tight_layout()
        plt.savefig(f'{MODELS}/feature_importance.png', dpi=100, bbox_inches='tight')
        plt.close()
        print("Saved feature_importance.png (no SHAP)")
    elif hasattr(clf, 'coef_'):
        coefs = pd.Series(np.abs(clf.coef_[0]), index=X.columns)
        top15 = coefs.nlargest(15)
        fig, ax = plt.subplots(figsize=(10, 8))
        top15.sort_values().plot(kind='barh', ax=ax, color='steelblue')
        ax.set_title('Feature Coefficients (Logistic Regression)')
        plt.tight_layout()
        plt.savefig(f'{MODELS}/feature_importance.png', dpi=100, bbox_inches='tight')
        plt.close()
        print("Saved feature_importance.png (coefficients)")


Saved: shap_global_summary.png


Saved: shap_dot_summary.png


In [4]:

if HAS_SHAP and shap_vals is not None:
    # ── Local Force Plot — single employee ──
    # Pick the highest-risk employee in test set
    y_prob = pipeline.predict_proba(X)[:, 1]
    high_risk_idx = np.argmax(y_prob)
    print(f"Generating local explanation for employee at index {high_risk_idx}")
    print(f"  Predicted attrition probability: {y_prob[high_risk_idx]:.4f}")
    
    # Waterfall plot (SHAP >= 0.40 API)
    try:
        explanation = shap.Explanation(
            values=shap_vals[high_risk_idx],
            base_values=explainer.expected_value if not isinstance(explainer.expected_value, list) 
                        else explainer.expected_value[1],
            data=X_transformed.iloc[high_risk_idx].values,
            feature_names=X.columns.tolist()
        )
        plt.figure(figsize=(12, 6))
        shap.plots.waterfall(explanation, show=False)
        plt.title(f'Local SHAP Explanation — Employee idx={high_risk_idx}', fontsize=13)
        plt.tight_layout()
        plt.savefig(f'{MODELS}/shap_local_force.png', dpi=100, bbox_inches='tight')
        plt.close()
        print("Saved: shap_local_force.png")
    except Exception as e:
        print(f"Waterfall plot error: {e} — saving bar plot instead")
        local_shap = pd.Series(shap_vals[high_risk_idx], index=X.columns)
        top_local = local_shap.abs().nlargest(10).index
        fig, ax = plt.subplots(figsize=(10, 6))
        local_shap[top_local].sort_values().plot(kind='barh', ax=ax, color='coral')
        ax.set_title(f'Local SHAP — Employee idx={high_risk_idx} (Risk={y_prob[high_risk_idx]:.2f})')
        plt.tight_layout()
        plt.savefig(f'{MODELS}/shap_local_force.png', dpi=100, bbox_inches='tight')
        plt.close()
        print("Saved: shap_local_force.png")


Generating local explanation for employee at index 111
  Predicted attrition probability: 0.9988


Saved: shap_local_force.png


**SHAP explainability complete.** Global feature importance and local explanation plots saved.